# WS4 — GRPO Training (Final)
FarmSimulation hackathon notebook.

In [ ]:
!pip install -q unsloth vllm \
trl==0.22.2 \
transformers==4.56.2 \
huggingface_hub==0.34.0 \
openenv-core wandb

In [ ]:
import os
from huggingface_hub import login
login(os.environ.get("HF_TOKEN", "hf_token_here"))
import wandb
wandb.login(anonymous="allow")


In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit", # upgraded to 1.5B
    max_seq_length=2048,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=16,
    gpu_memory_utilization=0.6, # adjust for 1.5B
    enforce_eager=True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
"""
FarmSimulation — Reward Functions v3
=====================================
Architecture: Single-step action quality (primary) + format + light terminal bonus
Model: Qwen/Qwen2.5-1.5B-Instruct  (upgrade from 0.5B — see README below)

WHY THIS WORKS vs your v2:
- v2 ran a full 30-step rollout inside GRPO → 0.5B model collapses the farm every time
  → all samples score ≈ 0.01 → zero gradient → bouncing curves
- v3 scores the SINGLE action against the observable state using deterministic rules
  → every sample gets a meaningful ±signal → GRPO gets real gradients → curves grow

PAPER PATTERNS APPLIED (Masud et al. 2026 "Reward Engineering for RL in SE Tasks"):
  P1 → Verifiable, tool-grounded outcome reward  = state_action_reward()
  P2 → Hybrid: step-level primary + terminal bonus (weighted, calibrated scales)
  P3 → Granularity matches observability: step-level signal, NOT sparse terminal

REWARD COMPONENT SCALES (calibrated so GRPO normalization works):
  state_action_reward   : −1.0  to +1.0   (primary — fires every sample)
  format_reward         : −0.3  to +0.3   (JSON validity — fires every sample)
  diversity_penalty     :  0.0  to −0.2   (batch-level anti-wait — fires per batch)
  terminal_bonus        :  0.0  to +0.5   (5-step short rollout — fires every sample)
  ─────────────────────────────────────────────────────────
  Total range           : −1.5  to +2.0
"""

import re
import json
import torch
import textwrap
import sys
sys.path.append('..')

# ── optional: only needed for terminal_bonus ──────────────────────────────────
try:
    from server.tasks import grade_episode_detailed, EpisodeRecord
    GRADING_AVAILABLE = True
except ImportError:
    GRADING_AVAILABLE = False

SPACE_URL = "http://127.0.0.1:7860"

# ---------------------------------------------------------------------------
# SYSTEM PROMPT  (unchanged from your v2 — keep consistent)
# ---------------------------------------------------------------------------
SYSTEM_PROMPT = textwrap.dedent("""\
    You are an autonomous farm manager. Each turn you observe the farm state and pick ONE action.
    Goal: maximize net worth via survival, growth, and well-timed market sales.
    Priorities: keep crops alive (irrigate when moisture is low), plant when you have seeds,
    harvest when mature, sell when prices are above the 7-day average.
    Reply with EXACTLY one JSON object:
    {"action_type": "...", "plot_id": 0, "seed_type": "...", "quantity": 1}.
    Omit fields that don't apply.
    Valid action_type values: wait, buy_seeds, plant, irrigate, harvest, sell,
    pump_water, apply_fertilizer, spray_pesticide, pull_weeds, buy_plot, clear,
    write_journal, end_day.""")

# Valid actions the env accepts
VALID_ACTIONS = {
    "wait", "buy_seeds", "plant", "irrigate", "harvest", "sell",
    "pump_water", "apply_fertilizer", "spray_pesticide", "pull_weeds",
    "buy_plot", "clear", "write_journal", "end_day"
}


# ===========================================================================
# UTILITY: Parse action JSON from completion text
# ===========================================================================
def parse_action(text: str) -> dict:
    """Extract the first valid JSON object from completion text."""
    try:
        # Try direct parse first
        return json.loads(text.strip())
    except Exception:
        pass
    # Find first {...} block
    match = re.search(r'\{[^}]+\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except Exception:
            pass
    return {}


# ===========================================================================
# UTILITY: Parse farm state from observation text
# Returns a structured dict of key numerical features
# ===========================================================================
def parse_farm_state(prompt_text: str) -> dict:
    """
    Extract numerical farm state from the observation text in the prompt.
    All values default to None if not found — callers must handle None.

    Parses:
      - moisture values per plot (list of floats)
      - crop stages per plot (list of str: 'empty','seedling','growing','mature')
      - current price and 7-day average per crop type
      - water tank level (0-100%)
      - money / net worth
      - seeds in inventory
      - labor hours remaining today
    """
    state = {
        "moisture": [],          # list of floats per plot
        "crop_stages": [],       # list of str per plot
        "crop_health": [],       # list of floats per plot
        "price_current": {},     # crop_type -> float
        "price_7day_avg": {},    # crop_type -> float
        "price_trend": {},       # crop_type -> 'rising'|'falling'|'stable'
        "water_tank_pct": None,  # float 0-1
        "money": None,           # float
        "seeds": {},             # seed_type -> count
        "labor_hours": None,     # float remaining today
        "has_mature_crop": False,
        "has_dry_crop": False,   # moisture < 0.3
        "has_critical_crop": False,  # moisture < 0.15
        "has_saturated_crop": False, # moisture > 0.8
        "price_above_avg": {},   # crop_type -> bool
    }

    # ── moisture ─────────────────────────────────────────────────────────────
    moisture_matches = re.findall(
        r'(?:moisture|soil moisture)[:\s=]+([0-9.]+)', prompt_text, re.IGNORECASE)
    state["moisture"] = [float(v) for v in moisture_matches]

    # ── crop stages ───────────────────────────────────────────────────────────
    stage_matches = re.findall(
        r'(?:stage|growth)[:\s=]+(empty|seedling|growing|mature|harvested)',
        prompt_text, re.IGNORECASE)
    state["crop_stages"] = [s.lower() for s in stage_matches]

    # ── crop health ───────────────────────────────────────────────────────────
    health_matches = re.findall(
        r'(?:health)[:\s=]+([0-9.]+)', prompt_text, re.IGNORECASE)
    state["crop_health"] = [float(v) for v in health_matches]

    # ── water tank ────────────────────────────────────────────────────────────
    tank_match = re.search(
        r'(?:water tank|tank)[:\s=]+([0-9.]+)\s*%?', prompt_text, re.IGNORECASE)
    if tank_match:
        val = float(tank_match.group(1))
        state["water_tank_pct"] = val / 100.0 if val > 1.0 else val

    # ── money ─────────────────────────────────────────────────────────────────
    money_match = re.search(
        r'(?:money|cash|balance|net worth)[:\s=]+\$?([0-9,]+\.?[0-9]*)',
        prompt_text, re.IGNORECASE)
    if money_match:
        state["money"] = float(money_match.group(1).replace(',', ''))

    # ── labor hours ───────────────────────────────────────────────────────────
    labor_match = re.search(
        r'(?:labor hours?|hours? remaining|hours? left)[:\s=]+([0-9.]+)',
        prompt_text, re.IGNORECASE)
    if labor_match:
        state["labor_hours"] = float(labor_match.group(1))

    # ── market prices ─────────────────────────────────────────────────────────
    # Pattern: "Wheat: $12.50 (avg: $11.20, trend: rising)"
    price_matches = re.finditer(
        r'(\w+)[:\s=]+\$([0-9.]+)\s*\((?:avg|average)[:\s=]+\$([0-9.]+)'
        r'(?:,?\s*trend[:\s=]+(\w+))?\)',
        prompt_text, re.IGNORECASE)
    for m in price_matches:
        crop = m.group(1).lower()
        current = float(m.group(2))
        avg = float(m.group(3))
        trend = m.group(4).lower() if m.group(4) else "stable"
        state["price_current"][crop] = current
        state["price_7day_avg"][crop] = avg
        state["price_trend"][crop] = trend
        state["price_above_avg"][crop] = current > avg

    # ── seeds ─────────────────────────────────────────────────────────────────
    seed_matches = re.finditer(
        r'(\w+)\s+seeds?[:\s=]+(\d+)', prompt_text, re.IGNORECASE)
    for m in seed_matches:
        state["seeds"][m.group(1).lower()] = int(m.group(2))

    # ── derived booleans ─────────────────────────────────────────────────────
    state["has_mature_crop"] = "mature" in state["crop_stages"]
    state["has_dry_crop"] = any(m < 0.3 for m in state["moisture"])
    state["has_critical_crop"] = any(m < 0.15 for m in state["moisture"])
    state["has_saturated_crop"] = any(m > 0.8 for m in state["moisture"])

    return state


# ===========================================================================
# REWARD 1: STATE-ACTION QUALITY  (PRIMARY — P1 + P3 from paper)
# Range: −1.0 to +1.0
# Dense, step-level, deterministic, verifiable against observable state.
# No rollout required.
# ===========================================================================
def state_action_reward(prompts, completions, **kwargs) -> list[float]:
    """
    Score each (state, action) pair using deterministic agricultural rules.
    This is your primary training signal. It fires on every GRPO sample.

    Rules are grounded in the same agricultural science as your env
    (FAO-56 evapotranspiration logic, standard crop management):

    IRRIGATION rules:
      + Critical moisture (<0.15) + irrigate      → +1.0  (emergency rescue)
      + Dry moisture (0.15-0.30) + irrigate       → +0.7  (correct proactive)
      + Normal moisture (0.30-0.60) + irrigate    → +0.0  (neutral, not wrong)
      + Moist moisture (0.60-0.80) + irrigate     → −0.4  (wasteful)
      + Saturated (>0.80) + irrigate              → −1.0  (very wasteful)

    HARVEST rules:
      + Mature crop exists + harvest              → +1.0  (correct)
      + No mature crop + harvest                  → −0.5  (invalid/wasted action)

    SELL rules:
      + Price above 7-day avg + sell              → +0.8  (smart timing)
      + Price below 7-day avg + sell              → −0.3  (poor timing)
      + No price data available + sell            → +0.1  (neutral)

    WAIT rules:
      + Has critical/dry crops + wait             → −1.0  (neglect)
      + Has mature crops + wait                   → −0.5  (dangerous neglect)
      + Nothing to do (no crops, no seeds) + wait → +0.1  (acceptable)
      + Active crops, moisture OK + wait           → +0.2  (patience reward)

    PUMP_WATER rules:
      + Tank below 30% + pump                     → +0.6  (proactive)
      + Tank above 70% + pump                     → −0.3  (wasteful)

    PLANT rules:
      + Has seeds + empty plots + plant           → +0.5  (correct)
      + No seeds + plant                          → −0.5  (invalid)

    BUY_SEEDS:
      + Has money (>$20) + no seeds + buy_seeds   → +0.4
      + Already has seeds + buy_seeds             → −0.1  (unnecessary)

    APPLY_FERTILIZER / SPRAY_PESTICIDE / PULL_WEEDS:
      + On a plot with a growing/mature crop      → +0.3
      + On empty plot                             → −0.3  (wasted labor)

    END_DAY / WRITE_JOURNAL:
      + Always → 0.0  (neutral — neither rewarded nor penalized)
    """
    rewards = []

    for prompt, completion in zip(prompts, completions):
        try:
            prompt_text = prompt[-1]["content"] if isinstance(prompt, list) else str(prompt)
            completion_text = completion[0]["content"] if isinstance(completion, list) else str(completion)

            action = parse_action(completion_text)
            action_type = action.get("action_type", "").lower()
            state = parse_farm_state(prompt_text)

            reward = _score_action(action_type, action, state)
            rewards.append(float(reward))

        except Exception:
            rewards.append(0.0)

    return rewards


def _score_action(action_type: str, action: dict, state: dict) -> float:
    """Core deterministic scoring logic. Returns float in [−1.0, +1.0]."""

    # ── IRRIGATE ──────────────────────────────────────────────────────────────
    if action_type == "irrigate":
        if not state["moisture"]:
            return 0.0  # can't score without state info
        # Score against the specific plot if plot_id given, else worst moisture
        plot_id = action.get("plot_id")
        if plot_id is not None and plot_id < len(state["moisture"]):
            m = state["moisture"][plot_id]
        else:
            m = min(state["moisture"])  # assume irrigating the driest plot

        if m < 0.15:
            return 1.0   # emergency rescue
        elif m < 0.30:
            return 0.7   # proactive — correct
        elif m < 0.60:
            return 0.0   # neutral, not harmful
        elif m < 0.80:
            return -0.4  # wasteful
        else:
            return -1.0  # saturated — very wasteful

    # ── HARVEST ──────────────────────────────────────────────────────────────
    elif action_type == "harvest":
        if state["has_mature_crop"]:
            return 1.0
        elif state["crop_stages"]:
            return -0.5  # tried to harvest non-mature crop
        else:
            return -0.3  # no crops at all

    # ── SELL ─────────────────────────────────────────────────────────────────
    elif action_type == "sell":
        above_avg = state["price_above_avg"]
        if not above_avg:
            return 0.1  # no price data — neutral
        # If any crop is above average, selling is smart
        if any(above_avg.values()):
            return 0.8
        else:
            return -0.3  # all crops below average price

    # ── WAIT ─────────────────────────────────────────────────────────────────
    elif action_type == "wait":
        if state["has_critical_crop"]:
            return -1.0   # NEVER wait when crops are dying
        elif state["has_dry_crop"]:
            return -0.7   # very bad — crops need water
        elif state["has_mature_crop"]:
            return -0.5   # risky — mature crops can wither
        elif not state["crop_stages"] and not state["seeds"]:
            return 0.1    # nothing to do — acceptable idle
        else:
            return 0.2    # crops growing, moisture OK — patience reward

    # ── PUMP_WATER ───────────────────────────────────────────────────────────
    elif action_type == "pump_water":
        tank = state["water_tank_pct"]
        if tank is None:
            return 0.2  # no data — slightly reward proactive pumping
        if tank < 0.30:
            return 0.6   # tank low — correct proactive action
        elif tank < 0.70:
            return 0.2   # tank moderate — mild reward
        else:
            return -0.3  # tank full — wasteful

    # ── PLANT ────────────────────────────────────────────────────────────────
    elif action_type == "plant":
        has_seeds = bool(state["seeds"])
        has_empty = "empty" in state["crop_stages"] or not state["crop_stages"]
        if has_seeds and has_empty:
            return 0.5
        elif not has_seeds:
            return -0.5   # can't plant without seeds
        else:
            return 0.1    # has seeds but maybe no empty plot — neutral

    # ── BUY_SEEDS ────────────────────────────────────────────────────────────
    elif action_type == "buy_seeds":
        has_seeds = bool(state["seeds"])
        has_money = state["money"] is not None and state["money"] > 20
        if not has_seeds and has_money:
            return 0.4
        elif has_seeds:
            return -0.1   # already have seeds — mildly penalize
        else:
            return 0.2    # no info — neutral-positive

    # ── APPLY_FERTILIZER / SPRAY_PESTICIDE / PULL_WEEDS ─────────────────────
    elif action_type in ("apply_fertilizer", "spray_pesticide", "pull_weeds"):
        plot_id = action.get("plot_id")
        if plot_id is not None and plot_id < len(state["crop_stages"]):
            stage = state["crop_stages"][plot_id]
            if stage == "empty":
                return -0.3  # wasted action on empty plot
            else:
                return 0.3   # valid maintenance action
        elif state["crop_stages"]:
            return 0.2       # crops exist, assume valid
        else:
            return -0.2      # no crops — probably wasteful

    # ── BUY_PLOT ─────────────────────────────────────────────────────────────
    elif action_type == "buy_plot":
        money = state["money"]
        if money is not None and money > 100:
            return 0.3   # expansion when you can afford it
        elif money is not None and money < 50:
            return -0.4  # dangerous — low funds
        else:
            return 0.1

    # ── END_DAY / WRITE_JOURNAL / CLEAR ──────────────────────────────────────
    elif action_type in ("end_day", "write_journal", "clear"):
        return 0.0  # neutral

    # ── UNKNOWN / EMPTY ──────────────────────────────────────────────────────
    else:
        return -0.5  # penalize invalid/unrecognized actions


# ===========================================================================
# REWARD 2: FORMAT QUALITY  (Replaces your broken format_reward)
# Range: −0.3 to +0.3
# Strict JSON validity + action type validity + required field check.
# ===========================================================================
def format_reward(prompts, completions, **kwargs) -> list[float]:
    """
    +0.3 if completion is valid JSON with a known action_type and correct fields.
    +0.1 if valid JSON but action_type missing or unknown.
     0.0 if invalid JSON but some JSON-like structure present.
    −0.3 if completely malformed (no JSON found at all).

    Required fields per action_type:
      irrigate, harvest, apply_fertilizer, spray_pesticide,
      pull_weeds, clear → plot_id
      plant             → plot_id, seed_type
      sell              → quantity (optional but checked)
      buy_seeds         → seed_type, quantity
    """
    REQUIRED_FIELDS = {
        "irrigate": ["plot_id"],
        "harvest": ["plot_id"],
        "plant": ["plot_id", "seed_type"],
        "apply_fertilizer": ["plot_id"],
        "spray_pesticide": ["plot_id"],
        "pull_weeds": ["plot_id"],
        "clear": ["plot_id"],
        "buy_seeds": ["seed_type"],
    }

    rewards = []
    for completion in completions:
        try:
            text = completion[0]["content"] if isinstance(completion, list) else str(completion)
            action = parse_action(text)

            if not action:
                # No JSON found at all
                rewards.append(-0.3)
                continue

            action_type = action.get("action_type", "")
            if action_type not in VALID_ACTIONS:
                rewards.append(0.1)  # valid JSON but unknown action
                continue

            # Check required fields
            required = REQUIRED_FIELDS.get(action_type, [])
            has_all_required = all(field in action for field in required)

            if has_all_required:
                rewards.append(0.3)
            else:
                rewards.append(0.1)  # valid action but missing fields

        except Exception:
            rewards.append(-0.3)

    return rewards


# ===========================================================================
# REWARD 3: DIVERSITY PENALTY  (Batch-level anti-wait-camping)
# Range: 0.0 to −0.2
# Applied per-sample but computed across the full GRPO group.
# Directly addresses your Round 1 wait-exploit problem.
# ===========================================================================
def diversity_penalty(prompts, completions, **kwargs) -> list[float]:
    """
    If more than 60% of completions in this GRPO batch choose 'wait',
    all wait actions get penalized −0.2.
    Other actions in an over-wait batch get a small bonus +0.05.

    This breaks the wait-camping equilibrium without hard-banning wait
    (wait is sometimes correct — see state_action_reward).
    """
    actions = []
    for completion in completions:
        try:
            text = completion[0]["content"] if isinstance(completion, list) else str(completion)
            action = parse_action(text)
            actions.append(action.get("action_type", "unknown").lower())
        except Exception:
            actions.append("unknown")

    wait_fraction = actions.count("wait") / max(len(actions), 1)
    over_waiting = wait_fraction > 0.60

    rewards = []
    for a in actions:
        if over_waiting:
            if a == "wait":
                rewards.append(-0.2)
            else:
                rewards.append(0.05)  # small bonus for doing SOMETHING
        else:
            rewards.append(0.0)  # batch is diverse — no penalty

    return rewards


# ===========================================================================
# REWARD 4: TERMINAL BONUS  (Short 5-step rollout — replaces your 30-step)
# Range: 0.0 to +0.5
# Only runs 5 steps (not 30) to avoid 0.5B/1.5B model collapse.
# Uses caching to avoid redundant rollouts.
# ===========================================================================
_ROLLOUT_CACHE_V3 = {}

def terminal_bonus(prompts, completions, **kwargs) -> list[float]:
    """
    Runs a SHORT 5-step rollout to get a partial episode signal.
    Capped at 0.5 max so it doesn't dominate over state_action_reward.

    If grading is unavailable (server down), returns 0.0 gracefully.
    """
    if not GRADING_AVAILABLE:
        return [0.0] * len(completions)

    try:
        from server.farm_env_client import FarmEnvClient
    except ImportError:
        return [0.0] * len(completions)

    rewards = []
    task_ids = kwargs.get("task_id", [1] * len(completions))
    seeds = kwargs.get("noise_seed", [42] * len(completions))

    for completion, t_id, seed in zip(completions, task_ids, seeds):
        try:
            text = completion[0]["content"] if isinstance(completion, list) else str(completion)
            cache_key = (text[:100], t_id, seed)  # cache on first 100 chars

            if cache_key in _ROLLOUT_CACHE_V3:
                rewards.append(_ROLLOUT_CACHE_V3[cache_key])
                continue

            first_action = parse_action(text)
            env = FarmEnvClient(SPACE_URL)
            obs = env.reset(task_id=t_id, noise_seed=seed)
            obs = env.step(first_action)

            # ── Only 5 steps ─────────────────────────────────────────────────
            for _ in range(4):
                if obs.get("done", False):
                    break
                obs_text = obs.get("text_summary", obs.get("narrative_text", ""))
                # Use rule-based fallback action (not model inference)
                # This avoids the 0.5B model collapsing the rollout
                fallback_action = _rule_based_fallback(obs_text)
                obs = env.step(fallback_action)

            record_dict = obs.get("metadata", {}).get("episode_record")
            if record_dict:
                record = EpisodeRecord(**record_dict)
                grade = grade_episode_detailed(record)
                bonus = min(grade.get("score", 0.0) * 0.5, 0.5)
            else:
                bonus = 0.0

            _ROLLOUT_CACHE_V3[cache_key] = bonus
            rewards.append(bonus)

        except Exception:
            rewards.append(0.0)

    return rewards


def _rule_based_fallback(obs_text: str) -> dict:
    """
    Simple deterministic policy for filling rollout steps 2-5.
    Uses the same parse logic as state_action_reward to choose
    a reasonable action without running the LLM.
    Prevents the 0.5B model from destroying the farm in the rollout.
    """
    state = parse_farm_state(obs_text)

    if state["has_critical_crop"]:
        # Find the driest plot
        if state["moisture"]:
            driest = state["moisture"].index(min(state["moisture"]))
            return {"action_type": "irrigate", "plot_id": driest}

    if state["has_mature_crop"] and state["crop_stages"]:
        mature_idx = next(
            (i for i, s in enumerate(state["crop_stages"]) if s == "mature"), 0)
        return {"action_type": "harvest", "plot_id": mature_idx}

    if state["has_dry_crop"] and state["moisture"]:
        driest = state["moisture"].index(min(state["moisture"]))
        return {"action_type": "irrigate", "plot_id": driest}

    if state["water_tank_pct"] is not None and state["water_tank_pct"] < 0.3:
        return {"action_type": "pump_water"}

    if state["seeds"] and "empty" in state["crop_stages"]:
        empty_idx = state["crop_stages"].index("empty")
        seed_type = next(iter(state["seeds"]))
        return {"action_type": "plant", "plot_id": empty_idx, "seed_type": seed_type}

    return {"action_type": "wait"}


# ===========================================================================
# COMBINED REWARD  (Pass this to GRPOTrainer as reward_funcs list)
# ===========================================================================
def combined_reward(prompts, completions, **kwargs) -> list[float]:
    """
    Normalized combination of all reward components.
    Weights: state_action=1.0, format=1.0, diversity=1.0, terminal=1.0

    Call this as a single reward_func if your GRPOTrainer accepts one.
    OR pass the four functions separately as a list (preferred — see training script).
    """
    r_action = state_action_reward(prompts, completions, **kwargs)
    r_format = format_reward(prompts, completions, **kwargs)
    r_diversity = diversity_penalty(prompts, completions, **kwargs)
    r_terminal = terminal_bonus(prompts, completions, **kwargs)

    combined = []
    for ra, rf, rd, rt in zip(r_action, r_format, r_diversity, r_terminal):
        combined.append(ra + rf + rd + rt)

    return combined


In [ ]:
# Cell 6: Action Parser
import json
import re
from typing import Dict, Any

_VALID_ACTION_TYPES = {
    "wait", "buy_seeds", "plant", "irrigate", "harvest", "sell",
    "pump_water", "apply_fertilizer", "spray_pesticide", "pull_weeds",
    "buy_plot", "clear", "write_journal", "end_day",
}
_FALLBACK_ACTION = {"action_type": "wait"}
_JSON_RE = re.compile(r"\{[^{}]*\}", re.DOTALL)

def parse_action(text: str) -> Dict[str, Any]:
    def ensure_dict(obj):
        if isinstance(obj, str):
            return {"action_type": obj}
        if isinstance(obj, dict):
            if "action" in obj and isinstance(obj["action"], str):
                return {"action_type": obj["action"]}
            if "action_type" not in obj and "action" in obj and isinstance(obj["action"], dict):
                return obj["action"]
            return obj
        return dict(_FALLBACK_ACTION)

    if not text or not text.strip():
        return dict(_FALLBACK_ACTION)
    
    # 1. Try direct JSON parse
    try:
        obj = json.loads(text.strip())
        if isinstance(obj, list) and len(obj) > 0:
            obj = obj[0]
        return ensure_dict(obj)
    except Exception:
        pass
    
    # 2. Try regex to find { ... }
    for m in _JSON_RE.finditer(text):
        try:
            obj = json.loads(m.group(0))
            return ensure_dict(obj)
        except Exception:
            continue
            
    # 3. If raw string is a valid action type
    clean_text = text.strip().lower().replace('"', '').replace('{', '').replace('}', '')
    if clean_text in _VALID_ACTION_TYPES:
        return {"action_type": clean_text}

    return dict(_FALLBACK_ACTION)


In [ ]:
# Cell 7: Reward Functions (Gen 3 — Simulation-Based)
import torch
import textwrap
import sys
import os
import importlib.util
from typing import List, Dict, Any

SYSTEM_PROMPT = textwrap.dedent("""\
    You are an autonomous farm manager. Each turn you observe the farm state and pick ONE action.
    Goal: maximize net worth via survival, growth, and well-timed market sales.
    Priorities: keep crops alive (irrigate when moisture is low), plant when you have seeds, harvest when mature, sell when prices are above the 7-day average.
    Reply with EXACTLY one JSON object: {"action_type": "...", "plot_id": 0, "seed_type": "...", "quantity": 1}.
    Omit fields that don't apply. Valid action_type values: wait, buy_seeds, plant, irrigate, harvest, sell, pump_water, apply_fertilizer, spray_pesticide, pull_weeds, buy_plot, clear, write_journal, end_day.""")

# --- MANUAL DIRECT LOADER ---
def load_env_class():
    # 1. Check if 'server' is a directory in /data
    server_dir = os.path.join(os.getcwd(), 'server')
    if not os.path.exists(server_dir):
         # Try one level up if in /notebooks
         server_dir = os.path.join(os.path.dirname(os.getcwd()), 'server')
    
    if os.path.exists(server_dir):
        print(f"📂 Found server folder at: {server_dir}")
        files = os.listdir(server_dir)
        print(f"📄 Files inside server/: {files}")
        
        # Try to find the file regardless of exact naming
        for f in files:
            if f.endswith('.py') and ('farming' in f.lower() or 'env' in f.lower()):
                target = os.path.join(server_dir, f)
                print(f"🎯 Attempting to load from: {target}")
                spec = importlib.util.spec_from_file_location("farming_mod", target)
                mod = importlib.util.module_from_spec(spec)
                spec.loader.exec_module(mod)
                # Look for the class inside the module
                for attr in dir(mod):
                    if 'FarmingEnvironment' in attr:
                        return getattr(mod, attr)
    return None

FarmingEnvironment = load_env_class()
if FarmingEnvironment:
    print("✅ Success: FarmingEnvironment loaded!")
else:
    print("❌ ERROR: Could not find class in server folder.")
    # Emergency fallback
    class FarmingEnvironment:
        def reset(self, **kwargs): return {}
        def step(self, a): return {}, 0, True, False, {}

def get_sim_reward(prompts, completions, **kwargs) -> List[float]:
    rewards = []
    temp_env = FarmingEnvironment()
    for prompt, completion in zip(prompts, completions):
        text = _extract_text(completion)
        action = parse_action(text)
        act_type = action.get('action_type', 'wait')
        if act_type == 'wait':
            rewards.append(-0.2)
            continue
        try:
            temp_env.reset(task_id=1)
            obs, reward, done, _, info = temp_env.step(action)
            for _ in range(15):
                if done: break
                obs, r, done, _, _ = temp_env.step({'action_type': 'end_day'})
            final_net_worth = obs.get('net_worth', 1000)
            net_gain = (final_net_worth - 1000) / 100.0
            rewards.append(max(-1.0, min(2.0, net_gain)))
        except:
            rewards.append(-0.5)
    return rewards

def format_reward(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        text = _extract_text(completion)
        action = parse_action(text)
        if action.get('action_type') != 'wait':
            rewards.append(0.5)
        else:
            rewards.append(0.0)
    return rewards

print("Gen 3 Reward Functions Loaded (Simulation-Based)")


In [ ]:
# Cell 8: Build the prompt dataset
from datasets import Dataset

SPACE_URL = "https://athric-farmsim.hf.space"


print("Sampling 200 initial states from the environment...")
dataset_rows = []
env = FarmEnvClient(SPACE_URL)

try:
    for seed in range(200):
        obs = env.reset(task_id=1, noise_seed=seed)
        text_summary = obs.get("text_summary", obs.get("narrative_text", "(no summary)"))
        valid_actions = ", ".join(obs.get("valid_actions", []))
        
        user_msg = f"STATE:\n{text_summary}\n\nVALID ACTIONS THIS STEP: {valid_actions}\n\nRECENT HISTORY:\n(none)\n\nReply with one JSON action."
        
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg}
        ]
        
        formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        
        dataset_rows.append({
            "prompt": formatted_prompt,
            "task_id": 1,
            "noise_seed": seed
        })
except Exception as e:
    print(f"Make sure server is running at {SPACE_URL} -> Error: {e}")

if dataset_rows:
    prompt_dataset = Dataset.from_list(dataset_rows)
    print(f"Created prompt dataset with {len(prompt_dataset)} rows.")

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="./grpo_farm_v3",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_generations=8,
    max_prompt_length=1024,
    max_completion_length=128, # short actions are more stable
    learning_rate=5e-6,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    bf16=True,
    logging_steps=5,
    report_to="wandb",
    beta=0.001,
    temperature=0.9,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[state_action_reward, format_reward, diversity_penalty, terminal_bonus],
    args=training_args,
    train_dataset=dataset,
)

In [ ]:
# Cell 10: GRPOTrainer
from trl import GRPOTrainer

if 'prompt_dataset' in locals():
    trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[
            get_sim_reward,
            format_reward,
        ],
        args=training_args,
        train_dataset=prompt_dataset,
    )
    print("Starting GRPO training (Gen 3)...")
    trainer.train()
    print("Training complete!")
else:
    print("ERROR: prompt_dataset not found. Run Cell 8 first.")


In [ ]:
# Cell 11: Plotting
!pip install matplotlib pandas

import matplotlib.pyplot as plt
import pandas as pd
import os

os.makedirs("assets", exist_ok=True)
if 'trainer' in locals() and trainer.state.log_history:
    df = pd.DataFrame(trainer.state.log_history)
    reward_cols = [c for c in df.columns if 'reward' in c and 'margin' not in c]
    if reward_cols:
        reward_df = df.dropna(subset=reward_cols, how='all')
        plt.figure(figsize=(10, 6))
        for col in reward_cols:
            plt.plot(reward_df['step'], reward_df[col], label=col, marker='o', markersize=3)
        plt.title("GRPO Reward Evolution (Qwen 0.5B + FarmSim)")
        plt.xlabel("Update Step")
        plt.ylabel("Reward Value")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig("assets/grpo_rewards.png", dpi=300, bbox_inches='tight')
        print("Saved reward plot to assets/grpo_rewards.png")

In [ ]:
# Cell 12: Save and Push to Hub
HF_USERNAME = "Athric"
REPO_ID = f"{HF_USERNAME}/qwen-0.5b-farmsim-grpo-gen2"

print("Saving locally to grpo_farm_qwen_0_5b_gen2_final...")
model.save_pretrained("grpo_farm_qwen_0_5b_gen2_final")
tokenizer.save_pretrained("grpo_farm_qwen_0_5b_gen2_final")

try:
    model.push_to_hub(REPO_ID, use_auth_token=True)
    tokenizer.push_to_hub(REPO_ID, use_auth_token=True)
    print(f"Successfully pushed Gen 2 adapter to Hub! URL: https://huggingface.co/{REPO_ID}")
except Exception as e:
    print("Push failed:", e)


In [ ]:
# Cell 13: Metadata
import torch
import unsloth
import trl

print("="*50)
print("🌾 FARMSIMULATION GRPO PIPELINE METADATA")
print("="*50)
print(f"PyTorch Version:  {torch.__version__}")
print(f"Unsloth Version:  {unsloth.__version__}")
print(f"TRL Version:      {trl.__version__}")
print(f"GPU Used:         {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Max Steps:        {training_args.max_steps}")
print(f"Loss Type:        {training_args.loss_type}")
print(f"Hardware Opt:     vLLM Fast Inference + 8-bit AdamW")
print("="*50)